# REDGARDEN Arena Bot AI — Unsupervised Pretraining

Runs on Google Colab (T4 GPU, though a small model trains fine on CPU too). Unsupervised
pretrains a **small custom GPT-2-shaped model, from scratch** (S170-220 -- not a fine-tune of
the public 124M-param GPT-2-small any more) on the REDGARDEN arena AI training corpus --
`NORTHSTAR.md` §18.4's own next buildable step (S170-194, founder: "do the work to prepare
for unsupervised learning" / "target torch training on colab").

**Open this directly from GitHub — no upload needed:** Colab → File → Open notebook →
GitHub tab → `emilyspringerton/REDGARDEN` →
`notebooks/redgarden_gpt2_pretrain_colab.ipynb`.

**Why small, from scratch, not a GPT-2-small fine-tune (S170-220, founder: "we want to embed
the weights right into the c code... we can do it all with colab scripts running python to do
it all"):** GPT-2-small's real weights are ~497MB as raw float32 -- too large to reasonably
commit to this git repo every training run, and almost certainly too slow for real-time CPU
inference inside REDGARDEN's own C game loop. The default config here (4 layers, 128 dim, 4
heads) is small enough for both. The real cost: a from-scratch small model can't load GPT-2's
own public English-language pretrained weights (different dimensions entirely), so this loses
that transfer-learning warm start GPT-2-small fine-tuning would have had.

**This is genuinely unsupervised, not the later supervised fine-tune.** Next-token
prediction over raw `self`/`foe` arena state + action text (`packages/simulation/
arena_ai_bridge.c`'s own `arena_corpus_record()` output, one record per active hero per
tick) — no win/loss label, no reward signal, learning "what tends to happen next" across
every hero's replay data undifferentiated. The resulting checkpoint is the STARTING
WEIGHTS for `NORTHSTAR.md` §12 Phase E's own already-planned supervised, NORN-graded
fine-tune (Milestone 7+), not a finished game-playing policy by itself.

**Before running this:** build the corpus locally and sync it to Drive:
```bash
# after playing/running some real matches, so var/corpus/ has real data
python3 scripts/build_ai_corpus.py --min-records 1000
# then copy var/corpus/combined.jsonl to Drive as redgarden-corpus.jsonl
# under the DRIVE_FOLDER path the bootstrap cell below sets
```

**Weight-embed + git-sync (S170-220):** after training, the checkpoint is automatically
exported to the flat binary format `packages/common/gpt2_infer.c`'s ported C inference engine
expects (same format the sibling `gpt2-alpine-c` repo's own `convert_checkpoint.py` uses), then
-- if an SSH key is found -- committed and pushed straight to `origin/main` as
`weights/redgarden-arena-bot.bin`. Founder: "i will put the keys in MyDrive/.ssh" -- put a real
deploy/personal SSH private key (with push access to the REDGARDEN repo) at
`MyDrive/.ssh/id_ed25519` (or override the path via `REDGARDEN_DRIVE_SSH_KEY` in the bootstrap
cell below). No key found there → the export still happens and still saves to Drive, git-sync
just skips itself rather than failing the whole run.

**Not yet built:** wiring this trained model into the LIVE bot AI decision loop
(`arena_game.c`'s `bot_cast_kit_if_ready` or a generalization of it) -- this pipeline trains,
exports, and syncs the weights; it doesn't yet make any bot actually use them in a real match.

**Paste-once workflow**, same pattern the sibling `gpt2-alpine-c` repo's own notebook
already uses: the one code cell below is all you ever paste into Colab. Hit play, approve
the Drive OAuth prompt when it appears, and it handles everything — clones (or pulls)
REDGARDEN, then runs `scripts/colab_train.py` for the actual training. All training logic
lives in that script, in git, not in this notebook — when the training approach changes,
it ships as a commit, and the *same* bootstrap cell picks it up on the next run via
`git pull`. Nothing to re-paste, no cells to manually resync.

In [ ]:
# === REDGARDEN arena bot AI unsupervised pretrain — reusable bootstrap cell ===
# This cell is the only thing you ever need to paste into Colab. It mounts
# Drive (approve the OAuth prompt when it appears), then pulls the latest
# training logic from git and runs it. Future changes to how training works
# ship as commits to scripts/colab_train.py — re-running this same cell
# always executes the current version, no re-pasting required.

from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess

REPO_URL = 'https://github.com/emilyspringerton/REDGARDEN.git'
REPO_DIR = '/content/REDGARDEN'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

# Adjust DRIVE_FOLDER here only if your Drive layout differs from the default.
os.environ.setdefault('DRIVE_FOLDER', '/content/drive/MyDrive/redgarden-training')

subprocess.run(
    ['python3', 'scripts/colab_train.py'],
    cwd=REPO_DIR, check=True,
)

## Next Steps

After training:
1. Download `checkpoint-unsupervised-pretrain.tar.gz` (the full HF checkpoint, for the later
   supervised fine-tune stage) from Drive, and/or check `origin/main` for
   `weights/redgarden-arena-bot.bin` (the C-inference-ready export, auto-pushed if an SSH key
   was present).
2. Keep the `.tar.gz` checkpoint as the starting point for §12 Phase E's own later supervised,
   NORN-graded fine-tune stage (that stage starts the model from this checkpoint, not a fresh
   from-scratch config) — not wired up yet, a separate future pass.
3. `weights/redgarden-arena-bot.bin` is ready for `packages/common/gpt2_infer.c` to load
   (`gpt2_model_load_weights`) -- but nothing in the live game yet actually calls it during a
   real match. Wiring real model inference into the bot AI decision loop is separate, future
   work, not done by this notebook.
4. File a completion Apple:
   ```bash
   emily apples post -t completion -repo REDGARDEN "Arena AI unsupervised pretrain complete" "..."
   ```
5. Update the relevant `EMILY/BACKLOG.md` item.